# silver_medicao

Camada **Silver** da medição: `leituras`.

- **Entrada:** `bronze.tb_leituras` (e referência `silver.tb_medidores` / `silver.tb_unidades_consumidoras`)
- **Saída:** `silver.tb_leituras` e `silver.tb_leituras_quarentena`
- **Segue o Padrão de Limpeza:** passa pelos 9 passos na ordem (ler e contar → estrutura → tipos → domínios → duplicatas → consistência → integridade → checagens e reconciliação → gravar).
- **Alimenta:** `gold_receita`, `gold_kpis`, `gold_fila_perdas` e `gold_visao_360`.
- Regras levantadas no notebook `exploracao_medicao`.

In [0]:
%run ./funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")
NOTEBOOK = "silver_medicao"

In [0]:
# ---------- Regras de negócio e constantes: Leituras ----------

# Em leitura_anterior, além dos textos padrão de 'sem valor' (vazio, N/A, NULL, -), '-1' também significa 'sem leitura anterior'
VALORES_NULOS_LEITURA_ANT = VALORES_NULOS + ("-1",)

# Domínio de tipo_leitura (o mapear_dominio já padroniza caixa e acento)
MAPA_TIPO_LEITURA = {
    "NORMAL": "REAL",
    "LIDA": "REAL",
    "EFETIVA": "REAL",
    "ESTIMADO": "ESTIMADA",
    "MEDIA": "ESTIMADA",
    "MÉDIA": "ESTIMADA",
    "AUTOPROD": "AUTENTICADA",
}
TIPOS_LEITURA_PERMITIDOS = ["REAL", "ESTIMADA", "MEDIA", "AUTENTICADA", "NAO REALIZADA"]

# Janela máxima (em dias) após a instalação de um novo medidor para caracterizar a 1ª leitura pós-troca
# Nesses ~6.215 casos, leitura_anterior herdou o acumulado do medidor antigo, mas o medidor novo partiu do zero:
# portanto o consumo real desde a instalação é a própria leitura_atual (recuperando ~2,4 GWh em P1).
JANELA_DIAS_POS_TROCA = 45

# Tolerância numérica (em kWh) para comparar consumo apurado com (leitura_atual - leitura_anterior)
TOLERANCIA_KWH = 0.05

## Passo 2: Ler e contar

In [0]:
lt = ler_bronze(CATALOGO, "leituras")

N_LT_BRONZE = lt.count()
REMOVIDAS_LT = {}      # o que cada passo remover entra aqui, com a quantidade medida

COLUNAS_LT = [c for c in lt.columns if not c.startswith("_")]
COL_CONSUMO = next((c for c in ["consumo_kwh", "consumo", "consumo_apurado"] if c in COLUNAS_LT), "consumo_kwh")
COL_COMPETENCIA = next((c for c in ["mes_referencia", "competencia"] if c in COLUNAS_LT), "mes_referencia")
COL_DATA_LEITURA = next((c for c in ["data_leitura", "data_medicao"] if c in COLUNAS_LT), "data_leitura")

print(f"Bronze leituras: {N_LT_BRONZE:,} linhas | Colunas: {COLUNAS_LT}")

## Passo 3: Estrutura

In [0]:
from functools import reduce

# 3a. Cabeçalho do CSV repetido no meio dos arquivos anuais
lt = remover_cabecalho_repetido(lt)
REMOVIDAS_LT["cabecalho"] = N_LT_BRONZE - lt.count()
if REMOVIDAS_LT["cabecalho"]:
    registrar("leituras", "cabeçalho do CSV repetido no meio do arquivo",
              "remover_cabecalho_repetido", "REMOVIDO", REMOVIDAS_LT["cabecalho"])

# 3b. Espaços nas pontas (e texto só de espaços -> nulo)
tem_espaco_nas_pontas = reduce(lambda a, b: a | b,
                               [F.col(c).rlike(r"^\s|\s$") for c in COLUNAS_LT])
n_espacos = lt.filter(tem_espaco_nas_pontas).count()

lt = aparar_textos(lt)
registrar("leituras", "espaços no início/fim dos textos",
          "aparar_textos", "CORRIGIDO", n_espacos)

## Passo 4: Tipos

In [0]:
# 4a. Tratamento de valores ausentes e sentinela '-1' em leitura_anterior (~32,7 mil linhas em 4 formas: vazio, N/A, NULL e -1)
leitura_ant_limpa = valores_para_nulo("leitura_anterior", VALORES_NULOS_LEITURA_ANT)
n_sem_leitura_ant = lt.filter(F.col("leitura_anterior").isNull() | leitura_ant_limpa.isNull()).count()

lt = lt.withColumn("leitura_anterior", leitura_ant_limpa)
registrar("leituras", "leitura_anterior sem valor em 4 formas (vazio, N/A, NULL e -1)",
          "valores_para_nulo(VALORES_NULOS + ('-1',))", "CORRIGIDO", n_sem_leitura_ant)

# Proteção para demais colunas numéricas caso contenham marcadores padrão de nulo
for c_num in ["leitura_atual", COL_CONSUMO]:
    lt = lt.withColumn(c_num, valores_para_nulo(c_num))

# 4b. Conversão de datas (yyyy-MM-dd, dd/MM/yyyy, dd.MM.yyyy, MM-dd-yyyy)
n_data_formato = lt.filter(
    F.col(COL_DATA_LEITURA).isNotNull()
    & ~F.col(COL_DATA_LEITURA).rlike(r"^\d{4}-\d{2}-\d{2}$")
).count()

checar_parse(lt, [COL_DATA_LEITURA], parse_data, "leituras")
lt = lt.withColumn(COL_DATA_LEITURA, parse_data(COL_DATA_LEITURA))

registrar("leituras", f"{COL_DATA_LEITURA} em 4 formatos (yyyy-MM-dd, dd/MM/yyyy, dd.MM.yyyy, MM-dd-yyyy)",
          "parse_data", "CORRIGIDO", n_data_formato)

# 4c. Conversão de decimais (leitura_anterior, leitura_atual e consumo em decimal(14,2))
DECIMAIS_LT = ["leitura_anterior", "leitura_atual", COL_CONSUMO]
tem_virgula_ou_rs = reduce(
    lambda a, b: a | b,
    [F.col(c).isNotNull() & F.col(c).rlike(r"[,R\$]") for c in DECIMAIS_LT]
)
n_decimais_fmt = lt.filter(tem_virgula_ou_rs).count()

checar_parse(lt, DECIMAIS_LT, parse_decimal, "leituras")
lt = lt.withColumns({c: parse_decimal(c) for c in DECIMAIS_LT})

if n_decimais_fmt:
    registrar("leituras", "valores de leitura/consumo formatados com vírgula decimal",
              "parse_decimal(14,2)", "CORRIGIDO", n_decimais_fmt)

## Passo 5: Domínios

In [0]:
# 5a. Padronização de tipo_leitura (maiúsculas, sem acento e mapeamento de sinônimos)
if "tipo_leitura" in COLUNAS_LT:
    novo_tipo = mapear_dominio("tipo_leitura", MAPA_TIPO_LEITURA)
    n_tipo_leitura = lt.filter(~F.col("tipo_leitura").eqNullSafe(novo_tipo)).count()
    lt = lt.withColumn("tipo_leitura", novo_tipo)
    registrar("leituras", "tipo_leitura com grafias variadas (caixa, acento ou sinônimo)",
              "mapear_dominio(MAPA_TIPO_LEITURA)", "CORRIGIDO", n_tipo_leitura)

# 5b. Padronização das chaves e competência como proteção
cols_chave_texto = [c for c in ["id_leitura", "id_medidor", "id_uc", COL_COMPETENCIA] if c in COLUNAS_LT]
lt = lt.withColumns({c: padronizar_texto(c) for c in cols_chave_texto})

## Passo 6: Duplicatas

In [0]:
n_antes = lt.count()
lt = deduplicar(lt)
REMOVIDAS_LT["duplicatas"] = n_antes - lt.count()

registrar("leituras", "linhas duplicadas idênticas nas colunas de negócio",
          "deduplicar(df)", "REMOVIDO", REMOVIDAS_LT["duplicatas"])
print(f"Antes: {n_antes:,} | depois: {n_antes - REMOVIDAS_LT['duplicatas']:,}")

## Passo 7: Consistência e Recuperação de Consumos Negativos (Pós-Troca e Sinal Invertido)

In [0]:
# 7a. Causa 1 (~6.215 casos): 1ª leitura pós-troca de medidor (até 45 dias da instalação)
# leitura_anterior herdou o acumulado do medidor antigo e o novo partiu do zero -> consumo_kwh = leitura_atual
md_silver = spark.table(f"{CATALOGO}.silver.tb_medidores")
md_instalacao = md_silver.select("id_medidor", F.col("data_instalacao").alias("__data_instalacao_md"))
lt = lt.join(md_instalacao, on="id_medidor", how="left")

dias_pos_instalacao = F.datediff(F.col(COL_DATA_LEITURA), F.col("__data_instalacao_md"))
cond_pos_troca = (
    (F.col(COL_CONSUMO) < 0)
    & dias_pos_instalacao.between(0, JANELA_DIAS_POS_TROCA)
)

lt = criar_flag(lt, "flag_primeira_leitura_pos_troca", cond_pos_troca)
n_pos_troca = lt.filter(F.col("flag_primeira_leitura_pos_troca")).count()

lt = lt.withColumn(
    COL_CONSUMO,
    F.when(F.col("flag_primeira_leitura_pos_troca"), F.col("leitura_atual"))
     .otherwise(F.col(COL_CONSUMO))
).drop("__data_instalacao_md")

registrar(
    "leituras",
    "1ª leitura pós-troca de medidor com consumo negativo (subtraiu acumulado do medidor antigo)",
    "consumo_kwh = leitura_atual (medidor novo parte do zero) + flag_primeira_leitura_pos_troca",
    "CORRIGIDO",
    n_pos_troca
)

# 7b. Caso 2 (Tratável com prova física dos 2 ponteiros): subtração invertida (anterior - atual)
# Só recupera quando AMBAS as leituras existem, o medidor avançou (leitura_atual > leitura_anterior)
# e o módulo do consumo bate com (leitura_atual - leitura_anterior) dentro de 1 kWh de arredondamento.
# Casos sem leitura_anterior ou com regressão/divergência de ponteiros NÃO são alterados e vão para flag_consumo_negativo.
diferenca_leituras = F.col("leitura_atual") - F.col("leitura_anterior")
cond_sinal_invertido = (
    (F.col(COL_CONSUMO) < 0)
    & ~F.col("flag_primeira_leitura_pos_troca")
    & F.col("leitura_anterior").isNotNull()
    & (F.col("leitura_atual") > F.col("leitura_anterior"))
    & (F.abs(F.abs(F.col(COL_CONSUMO)) - diferenca_leituras) <= F.lit(1.0))
)
n_sinal_invertido = lt.filter(cond_sinal_invertido).count()

lt = lt.withColumn(
    COL_CONSUMO,
    F.when(cond_sinal_invertido, F.abs(F.col(COL_CONSUMO)))
     .otherwise(F.col(COL_CONSUMO))
)

registrar(
    "leituras",
    "consumo_kwh com sinal invertido comprovado pelos acumuladores (leitura_atual > leitura_anterior e |consumo| == delta)",
    f"{COL_CONSUMO} = abs({COL_CONSUMO})",
    "CORRIGIDO",
    n_sinal_invertido
)

# 7c. Demais flags de consistência da medição (booleanas, prefixo flag_, nunca nulas)
# Quem continuar com consumo_kwh < 0 (ex.: sem leitura_anterior ou regressão fora de troca) recebe flag_consumo_negativo = True
lt = criar_flag(lt, "flag_leitura_anterior_ausente", F.col("leitura_anterior").isNull())

if "tipo_leitura" in COLUNAS_LT:
    lt = criar_flag(lt, "flag_leitura_estimada", F.col("tipo_leitura").isin("ESTIMADA", "MEDIA"))

lt = criar_flag(lt, "flag_consumo_negativo", F.col(COL_CONSUMO) < 0)

lt = criar_flag(
    lt,
    "flag_consumo_inconsistente",
    F.col("leitura_anterior").isNotNull()
    & ~F.col("flag_primeira_leitura_pos_troca")
    & (F.abs(F.col(COL_CONSUMO) - diferenca_leituras) > F.lit(TOLERANCIA_KWH))
)

FLAGS_LT = {
    "flag_primeira_leitura_pos_troca": (
        "1ª leitura até 45 dias após troca de medidor (consumo recuperado via leitura_atual)",
        "consumo_kwh < 0 e datediff(data_leitura, data_instalacao) between 0 and 45"
    ),
    "flag_leitura_anterior_ausente": (
        "leitura_anterior ausente (1ª leitura do medidor ou valor não informado)",
        "leitura_anterior is null"
    ),
}
if "tipo_leitura" in COLUNAS_LT:
    FLAGS_LT["flag_leitura_estimada"] = (
        "leitura realizada por estimativa/média",
        "tipo_leitura in ('ESTIMADA', 'MEDIA')"
    )
FLAGS_LT["flag_consumo_negativo"] = (
    "consumo apurado negativo remanescente (regressão sem troca de medidor)",
    f"{COL_CONSUMO} < 0"
)
FLAGS_LT["flag_consumo_inconsistente"] = (
    "consumo apurado diverge de (leitura_atual - leitura_anterior)",
    f"abs({COL_CONSUMO} - (leitura_atual - leitura_anterior)) > {TOLERANCIA_KWH}"
)

# Conta todas as flags numa única leitura e registra no log
contagem_flags = lt.agg(*[F.sum(F.col(f).cast("int")).alias(f) for f in FLAGS_LT]).first().asDict()
for flag, (problema, regra) in FLAGS_LT.items():
    registrar("leituras", problema, regra, "FLAG", contagem_flags[flag])

## Passo 8: Integridade referencial

In [0]:
# Regra do contrato: ligar leituras ao cadastro por id_medidor contra silver.tb_medidores
lt, lt_quarentena = separar_orfaos(
    lt,
    md_silver,
    "id_medidor",
    motivo="id_medidor inexistente em silver.tb_medidores"
)

# Caso a tabela também traga id_uc explícito, garante integridade também contra silver.tb_unidades_consumidoras
if "id_uc" in COLUNAS_LT:
    uc_silver = spark.table(f"{CATALOGO}.silver.tb_unidades_consumidoras")
    lt, lt_orfaos_uc = separar_orfaos(
        lt,
        uc_silver,
        "id_uc",
        motivo="id_uc inexistente em silver.tb_unidades_consumidoras"
    )
    lt_quarentena = lt_quarentena.unionByName(lt_orfaos_uc)

n_orfaos = lt_quarentena.count()
registrar("leituras", "leitura de medidor/UC inexistente no cadastro (inclui UCs de teste)",
          "separar_orfaos vs silver.tb_medidores", "QUARENTENA", n_orfaos)

## Passo 9: Checagens e reconciliação (antes de gravar)

In [0]:
# Se qualquer checagem ou a reconciliação falhar, o notebook para ANTES de sobrescrever a Silver.
chave_unicidade = ["id_leitura"] if "id_leitura" in COLUNAS_LT else ["id_medidor", COL_COMPETENCIA]
checar_unicidade(lt, chave_unicidade, "leituras")

# Como recuperamos o consumo_kwh (em vez de anular), COL_CONSUMO continua 100% sem nulos!
cols_obrigatorias = [c for c in ["id_leitura", "id_uc", "id_medidor", COL_COMPETENCIA, COL_DATA_LEITURA, "leitura_atual", COL_CONSUMO] if c in COLUNAS_LT]
checar_sem_nulos(lt, cols_obrigatorias, "leituras")

if "tipo_leitura" in COLUNAS_LT:
    checar_dominio(lt, "tipo_leitura", TIPOS_LEITURA_PERMITIDOS, "leituras")

reconciliar("leituras", N_LT_BRONZE, lt.count(), lt_quarentena.count(), REMOVIDAS_LT)

## Passo 10: Gravar Silver

In [0]:
# Ordem final das colunas conforme convenção do time: negócio -> flags -> rastreio
lt = lt.select(*COLUNAS_LT, *FLAGS_LT, "_source_file", "_ingestion_timestamp")

n_lt = gravar_silver(lt, CATALOGO, "leituras", quarentena=lt_quarentena)

spark.table(f"{CATALOGO}.silver.tb_leituras").printSchema()

## Fechamento: log de qualidade

In [0]:
gravar_log(CATALOGO, NOTEBOOK)

display(spark.table(f"{CATALOGO}.silver.tb_log_qualidade")
             .filter(F.col("notebook") == NOTEBOOK)
             .orderBy("tabela", "destino"))